# LLM-as-a-Judge
Evaluates a mental-health trajectory report against the subject's post history using GPT as an impartial judge.
Scores are produced on a 5-point Likert scale across six criteria and saved as JSON.

In [ ]:
# =========================
# IMPORTS
# =========================
import json
from pathlib import Path
from openai import OpenAI
import pandas as pd

In [ ]:
# =========================
# CONFIG
# =========================
USER        = "2714"
MODEL_NAME  = "gpt-4o"
TEMPERATURE = 0.0
REPORT_TYPE = "E"   # A -> base report  |  E -> trajectory report

APIKEY = ""

# Paths
REPORT_DIR = r"C:\Users\Franco\Desktop\Pubblicazione\15\report"
DATA_DIR   = r"C:\Users\Franco\Desktop\Pubblicazione\15\data"
OUT_DIR    = r"C:\Users\Franco\Desktop\Pubblicazione\15\judge"

REPORT_PATH       = rf"{REPORT_DIR}\{USER}_report_{REPORT_TYPE}.txt"
DATA_CLEANED_PATH = rf"{DATA_DIR}\{USER}_data_cleaned.csv"
OUTPUT_PATH       = rf"{OUT_DIR}\{USER}_judge_GPT_{REPORT_TYPE}.json"

In [ ]:
# =========================
# HELPERS
# =========================
def read_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def format_post_history(csv_path):
    df = pd.read_csv(csv_path, encoding="utf-8")
    # Ensure we only keep Date and Text, formatted for the prompt
    if "Date" not in df.columns or "Text" not in df.columns:
        raise ValueError(f"File {csv_path} must contain 'Date' and 'Text' columns")
    formatted_posts = []
    for _, row in df.iterrows():
        formatted_posts.append(f"[{row['Date']}] {row['Text']}")
    return "\n".join(formatted_posts)

In [ ]:
# =========================
# LOAD DATA
# =========================
print(f"Loading report for User {USER}...")
report_content = read_file(REPORT_PATH)

print(f"Loading post history for User {USER}...")
post_history_content = format_post_history(DATA_CLEANED_PATH)

print(f"Report length   : {len(report_content.split())} words")
print(f"Post history    : {len(post_history_content.splitlines())} posts")

In [ ]:
# =========================
# PROMPTING
# =========================
system_prompt = """You are an expert, impartial evaluator of mental-health trajectory reports.
Your task is to assess how well a report characterizes the depressive evolution of a subject, based solely on their post history.
Evaluate only what is explicitly supported by the data. Output only valid JSON."""

user_prompt = f"""You are provided with two sections of text below:
POST HISTORY (GROUND TRUTH) — the subject's chronological post history, formatted line by line as [Date] Post Text. This is your sole ground truth.
REPORT TO EVALUATE — the report to be evaluated.
Read the post history carefully and use it as the reference for all evaluations.
Evaluate the report using a 5-point Likert scale (1 = Very poor, 5 = Excellent) strictly based on the Evaluation Grid below.
Important: Evaluate the report based on its accuracy and grounding in the source material, regardless of its length. The report is permitted to make valid clinical inferences derived directly from the subject's behaviors or statements in the posts, but it must never make up facts, fabricate events, or introduce unsupported claims. The primary purpose of this report is to accurately characterize the depressive evolution of the subject as evidenced in their posts.


CRITERIA
Factual Accuracy
Trajectory Coverage
Temporal Coherence
Sensitivity to Change Points
Segment-Level Specificity
Overall Preference


CRITERIA DEFINITIONS
Factual Accuracy: the degree to which the claims made in the report are directly and verifiably supported by the content of the posts. Penalizes fabrications, distortions, over-interpretations, or omissions of clearly documented events.
Trajectory Coverage: how well the report captures the main phases of the subject's depressive history as evidenced in the posts, rather than focusing only on isolated posts or a single time period. Coverage should be judged relative to what the data actually contains, not rewarded for inventing phases not supported by the data.
Temporal Coherence: the extent to which the report accurately and clearly describes how the subject's state changed over time (e.g., worsening, improvement, stability), maintaining a logically consistent narrative that reflects the actual chronological sequence of posts.
Sensitivity to Change Points: how clearly and accurately the report identifies and explains important turning points in the trajectory — such as transitions from high to lower severity or vice versa — as they are genuinely documented in the post history.
Segment-Level Specificity: the degree to which the report provides concrete, phase-specific details (e.g., recurring themes, expressed emotions, coping behaviors, language patterns) that are grounded in the actual posts and meaningfully characterize each phase — not generic descriptors that could apply to any subject.
Overall Preference: an overall judgment of which report provides the most useful and coherent description of the user's depression-related trajectory.


EVALUATION GRID (RUBRIC)
1. Factual Accuracy
1: Report contains multiple claims that directly contradict or misrepresent the posts; fabrications present.
2: Several inaccuracies or unsupported interpretations that distort the overall picture.
3: Mostly accurate, but includes some over-interpretations, unsupported generalizations, or minor misrepresentations.
4: Accurate throughout; all major claims are supported by the posts, with only negligible imprecision.
5: Perfectly accurate; every claim is directly traceable to specific post content, with no distortion or fabrication.
2. Trajectory Coverage
1: Focuses exclusively on a single event or time period; fails entirely to capture a trajectory.
2: Mentions more than one period but leaves massive gaps; highly fragmented relative to the available data.
3: Captures a basic outline of the history but misses at least one major phase that is clearly present in the data.
4: Captures most data-supported phases well, providing a mostly complete and accurate historical overview.
5: Comprehensively and accurately maps all distinct phases evidenced in the data from start to finish, without notable gaps or invented phases.
3. Temporal Coherence
1: Narrative is chaotic or contradicts the actual chronological order of posts.
2: A timeline is attempted, but contradictions, anachronisms, or confusing transitions make it hard to follow.
3: Chronology is generally understandable and consistent with the data, but flows poorly or has minor logical gaps.
4: Clear and accurate timeline; changes over time are well-described and reflect the actual post sequence, with only minor awkward transitions.
5: Exceptionally logical, seamless, and chronologically faithful narrative; temporal evolution is perfectly tracked and matches the data.
4. Sensitivity to Change Points
1: Ignores turning points completely; treats the trajectory as entirely flat or static despite evidence of change in the data.
2: Vaguely alludes to changes but fails to identify specific turning points or misattributes shifts not supported by the data.
3: Identifies major turning points documented in the data but lacks depth or accuracy in explaining the nature of the transition.
4: Clearly and accurately identifies and explains most turning points; transitions in severity are well-grounded in the post content.
5: Precisely and accurately identifies every critical shift documented in the data, explaining its nature and timing with full fidelity to the source.
5. Segment-Level Specificity
1: Entirely abstract or vague; no concrete details; could describe any subject.
2: Mentions a few general themes but relies on broad, non-subject-specific summaries without grounding in the actual posts.
3: Provides some concrete, data-grounded details for certain phases, but coverage is uneven or some phases remain generic.
4: Good level of accurate, subject-specific detail; most phases include references to themes, emotions, or behaviors actually documented in the posts.
5: Highly granular and phase-specific; every segment is characterized by accurate, concrete, and meaningful details drawn directly from the post content.
6. Overall Preference
1: Extremely poor and unusable; provides no useful characterization of the trajectory.
2: Weak and generally unhelpful; major flaws outweigh utility.
3: Adequate but unexceptional; provides a basic but flawed or incomplete description.
4: Strong, useful, and clearly well-constructed; highly coherent description of the user's depression-related trajectory.
5: Exceptional, standing out as the definitive, most useful, and flawlessly coherent description of the trajectory.


OUTPUT FORMAT (STRICT JSON ONLY)
{{
  "Report_Evaluation": {{
    "Factual_Accuracy": X,
    "Trajectory_Coverage": X,
    "Temporal_Coherence": X,
    "Change_Point_Sensitivity": X,
    "Segment_Level_Specificity": X,
    "Overall_Preference": X
  }},
  "Criterion_Justifications": {{
    "Factual_Accuracy": "Concise justification referencing the rubric and specific post evidence (2 sentences).",
    "Trajectory_Coverage": "Concise justification referencing the rubric and the phases present in the data (2 sentences).",
    "Temporal_Coherence": "Concise justification referencing the rubric and the actual post chronology (2 sentences).",
    "Change_Point_Sensitivity": "Concise justification referencing the rubric and specific turning points in the data (2 sentences).",
    "Segment_Level_Specificity": "Concise justification referencing the rubric and data-grounded details (2 sentences).",
    "Overall_Preference": "Concise overall justification of the report's utility and coherence (2 sentences)."
  }},
  "Rationale": "Concise overall summary of the evaluation: how well the report captures and characterizes the depressive evolution of the subject as documented in their posts (2–4 sentences)."
}}

POST HISTORY (GROUND TRUTH):
<<<
{post_history_content}
>>>

REPORT TO EVALUATE:
<<<
{report_content}
>>>
"""

print("Prompts ready.")

In [ ]:
# =========================
# SUBMIT TO LLM
# =========================
print("Calling the LLM...")
client = OpenAI(api_key=APIKEY)

response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ],
)

judge_output = response.choices[0].message.content
judge_json   = json.loads(judge_output)

print("\n=== LLM-as-a-Judge Output ===\n")
print(json.dumps(judge_json, indent=2, ensure_ascii=False))

In [ ]:
# =========================
# SAVE OUTPUT
# =========================
# Ensure the output directory exists before saving
output_dir = Path(OUTPUT_PATH).parent
output_dir.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(judge_json, f, indent=2, ensure_ascii=False)

print(f"\n✅ Output saved to {OUTPUT_PATH}")